In [ ]:
# In colab run this cell first to setup the file structure!
%cd /content
!rm -rf MOL518-Intro-to-Data-Analysis

!git clone https://github.com/shaevitz/MOL518-Intro-to-Data-Analysis.git
%cd MOL518-Intro-to-Data-Analysis/Lecture_36

# BPY518 Lecture 36: Image Segmentation

## Lecture Outline
- Binary masks and segmentation pipelines
- Morphological operations and structuring elements
- Labeling connected components
- Measuring object properties and filtering objects
- Skeletons and boundaries
- Watershed splitting for touching objects

Segmentation is the process of dividing an image into meaningful regions: cells, nuclei, vesicles, tissues, or any other objects we want to count and measure.

A typical segmentation pipeline is:

1. Preprocess the image.
2. Threshold to get a binary mask.
3. Clean that mask with morphological operations.
4. Label the objects.
5. Measure and filter objects.
6. Split touching objects if needed.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from skimage import color, filters, measure, morphology, segmentation

plt.rcParams['figure.dpi'] = 120
plt.rcParams['image.cmap'] = 'gray'

media_dir = Path('media/Lecture_6')
rng = np.random.default_rng(12)

## Binary Masks

The first major goal in segmentation is to turn a grayscale image into a **binary mask**: foreground pixels are `1` and background pixels are `0`.

The figure below illustrates that progression on an E. coli image:

![Original image, binary mask, and labeled objects](media/labeled_objects_example.png)

A mask is powerful because it turns image analysis into region analysis: once we know which pixels belong to a cell, we can label, count, and measure those pixels.

Let's make sample image and use the Otsu method to threshold it.

In [ ]:
def make_segmentation_demo(shape=(180, 180), rng=None):
    yy, xx = np.mgrid[: shape[0], : shape[1]]
    image = 0.15 + 0.02 * np.sin(xx / 17) + 0.02 * np.cos(yy / 23)
    centers = [(42, 36), (55, 118), (92, 58), (118, 122), (142, 42)]
    radii = [10, 14, 12, 15, 11]
    amplitudes = [0.75, 0.9, 0.82, 0.88, 0.7]
    for (cy, cx), radius, amplitude in zip(centers, radii, amplitudes):
        image += amplitude * np.exp(-((xx - cx) ** 2 + (yy - cy) ** 2) / (2 * radius**2))

    # Add artifacts directly in the image
    image += 0.65 * np.exp(-((xx - 145) ** 2) / (2 * 5**2)) * np.exp(-((yy - 92) ** 2) / (2 * 18**2))
    image += 0.38 * np.exp(-((xx - 151) ** 2 + (yy - 23) ** 2) / (2 * 2.8**2))
    image += 0.34 * np.exp(-((xx - 19) ** 2 + (yy - 146) ** 2) / (2 * 2.6**2))
    image -= 0.8 * np.exp(-((xx - 117) ** 2 + (yy - 61) ** 2) / (2 * 4.8**2))

    if rng is not None:
        image += 0.03 * rng.normal(size=shape)
    return np.clip(image, 0, 1)


demo_image = make_segmentation_demo(rng=rng)
smoothed = ndi.gaussian_filter(demo_image, sigma=1.2)
threshold = filters.threshold_otsu(smoothed)
initial_mask = smoothed > threshold #make the binary mask

fig, axes = plt.subplots(1, 3, figsize=(11, 3.8))
axes[0].imshow(demo_image)
axes[0].set_title('Input image')
axes[0].axis('off')

axes[1].imshow(smoothed)
axes[1].set_title('Smoothed image')
axes[1].axis('off')

axes[2].imshow(initial_mask)
axes[2].set_title('Initial binary mask')
axes[2].axis('off')

plt.tight_layout()
print(f"Otsu threshold: {threshold:.3f}")

## Morphological Operations

Morphological operations modify the shape and connectivity of objects in a binary mask. The main ones are:

- **Erosion** shrinks foreground objects.
- **Dilation** expands them.
- **Opening** = erosion then dilation.
- **Closing** = dilation then erosion.
- **Hole filling** repairs internal gaps.

These operations are defined by a **structuring element**: a small neighborhood such as a disk, square, or cross. Different structuring elements favor different geometric effects.

In [ ]:
# Clean the mask with common morphological operations.
disk = morphology.disk(5)

eroded = ndi.binary_erosion(initial_mask, structure=disk)
dilated = ndi.binary_dilation(initial_mask, structure=disk)
opened = ndi.binary_opening(initial_mask, structure=disk)
closed = ndi.binary_closing(initial_mask, structure=disk)
filled = ndi.binary_fill_holes(closed)

fig, axes = plt.subplots(2, 3, figsize=(10, 7))
examples = [
    ('Input mask', initial_mask),
    ('Erosion', eroded),
    ('Dilation', dilated),
    ('Opening', opened),
    ('Closing', closed),
    ('Fill holes', filled),
]
for ax, (title, image) in zip(axes.flat, examples):
    ax.imshow(image)
    ax.set_title(title)
    ax.axis('off')

plt.tight_layout()

Opening is often useful for removing small specks and breaking thin bridges. Closing is useful for sealing small gaps and smoothing contours. Hole filling is especially common once we are confident that object interiors should be solid.

## Labeling Connected Components

Once a mask is clean enough, we can turn it into explicit objects by assigning each `connected component` its own integer label.

In [ ]:
labeled_mask, num_objects = ndi.label(filled)
label_rgb = color.label2rgb(labeled_mask, bg_label=0, bg_color=(0, 0, 0))

fig, axes = plt.subplots(1, 2, figsize=(8.5, 4))
axes[0].imshow(filled)
axes[0].set_title('Clean binary mask')
axes[0].axis('off')

axes[1].imshow(label_rgb)
axes[1].set_title(f'Labeled objects ({num_objects} found)')

axes[1].axis('off')
plt.tight_layout()

## Measuring Region Properties

After labeling, we can compute features for each object: area, centroid, perimeter, eccentricity, orientation, intensity, and more. If these were cells, area would tell you about the cell area/volume etc.

In [ ]:
props_table = measure.regionprops_table(
    labeled_mask,
    intensity_image=smoothed,
    properties=('label', 'area', 'centroid', 'eccentricity', 'perimeter', 'mean_intensity'),
)
props_df = pd.DataFrame(props_table)
props_df.round(2) 

## Filtering Objects by Property
We often want to keep only objects with plausible size or shape.

For example, if we want roughly cell-like objects, we might require a minimum area and reject very elongated artifacts.

In [ ]:
# Keep only objects that look like circles of a reasonable size.
keep_labels = props_df.loc[
    (props_df['area'] > 250) & (props_df['eccentricity'] < 0.85),
    'label',
]
filtered_mask = np.isin(labeled_mask, keep_labels)
filtered_labels, filtered_count = ndi.label(filtered_mask)

fig, axes = plt.subplots(1, 3, figsize=(11, 3.8))
axes[0].imshow(label_rgb)
axes[0].set_title('All labeled objects')
axes[0].axis('off')

axes[1].imshow(filtered_mask)
axes[1].set_title('Filtered mask')
axes[1].axis('off')

axes[2].imshow(color.label2rgb(filtered_labels, bg_label=0, bg_color=(0, 0, 0)))
axes[2].set_title(f'Filtered labels ({filtered_count} kept)')
axes[2].axis('off')

plt.tight_layout()
print('Kept labels:', keep_labels.tolist())

### Exercise 1 (If we have time...)

Go back up in the notebook and try to revise the mask so it splits the two circles on the right and then rerun the analysis.

## Skeletons and Boundaries

Sometimes we do not need the full mask area. Instead we may want:

- a **skeleton**, that traces the centerline along each object
- the **boundary**, which traces the outer contour of each object

The examples below show both representations on the E. coli masks we saw in the beginning of the lecture.

![Skeleton example](media/skeleton_example.png)

![Boundary overlay example](media/boundary_overlay_example.png)

In [ ]:
# Extract skeletons and boundaries from the filtered mask.
skeleton = morphology.skeletonize(filtered_mask)
boundaries = segmentation.find_boundaries(filtered_mask, mode='outer')

boundary_overlay = np.dstack([smoothed, smoothed, smoothed])
boundary_overlay[..., 0] = np.maximum(boundary_overlay[..., 0], boundaries.astype(float))
boundary_overlay[..., 1] *= ~boundaries
boundary_overlay[..., 2] *= ~boundaries

fig, axes = plt.subplots(1, 3, figsize=(11, 3.8))
axes[0].imshow(filtered_mask)
axes[0].set_title('Filtered mask')
axes[0].axis('off')

axes[1].imshow(skeleton)
axes[1].set_title('Skeleton')
axes[1].axis('off')

axes[2].imshow(boundary_overlay)
axes[2].set_title('Boundaries overlaid')
axes[2].axis('off')

plt.tight_layout()

## Touching Objects and Watershed Splitting

Thresholding alone often merges nearby cells into one connected component. The classic fix is a watershed workflow:

1. Compute the distance transform inside the binary mask.
2. Find local maxima that serve as object centers.
3. Use those maxima as markers.
4. Run watershed on the negative distance map.

![Watershed algorithm overview](media/watershed_algorithm_overview.png)

The examples below show the stages of that pipeline:

![Distance transform example](media/watershed_distance_transform.png)

![Watershed markers](media/watershed_markers.png)

![Final watershed segmentation](media/watershed_final.png)

In [ ]:
# Build a synthetic example with two touching cells and split them by watershed.
yy, xx = np.mgrid[:180, :180]
cell_mask_1 = (xx - 74) ** 2 + (yy - 90) ** 2 <= 20**2
cell_mask_2 = (xx - 106) ** 2 + (yy - 90) ** 2 <= 20**2
touching_mask = cell_mask_1 | cell_mask_2
touching_image = ndi.gaussian_filter(touching_mask.astype(float), sigma=2.5)
touching_image += 0.01 * np.random.default_rng(123).normal(size=touching_image.shape)
touching_image = np.clip(touching_image, 0, None)

mask_touching = touching_image > filters.threshold_otsu(touching_image)
distance = ndi.distance_transform_edt(mask_touching)
peak_response = ndi.maximum_filter(distance, size=21)
local_max = (distance == peak_response) & (distance > 8) #This is our local max finder from last lecture
markers, marker_count = ndi.label(local_max)
watershed_labels = segmentation.watershed(-distance, markers, mask=mask_touching)
watershed_count = len(np.unique(watershed_labels)) - 1

fig, axes = plt.subplots(2, 3, figsize=(10, 7))
views = [
    ('Touching cells', touching_image),
    ('Binary mask', mask_touching),
    ('Distance transform', distance),
    ('Local maxima markers', local_max),
    ('Marker labels', markers),
    ('Watershed result', color.label2rgb(watershed_labels, bg_label=0, bg_color=(0, 0, 0))),
]
for ax, (title, image) in zip(axes.flat, views):
    ax.imshow(image)
    ax.set_title(title)
    ax.axis('off')

plt.tight_layout()